In [4]:
from collections import Counter
import math

In [5]:
class SinglePassClustering:

  def __init__(self, threshold=0.25):
    self.threshold = threshold
    self.clusters = []  # List of dicts representing cluster centroids
    self.cluster_docs = []  # List of lists containing doc IDs per cluster

  def _text_to_vector(self, text):
    """Converts text into a term frequency vector (dictionary)."""
    words = text.lower().split()
    return Counter(words)

  def _cosine_similarity(self, vec1, vec2):
    """Calculates cosine similarity between two term frequency vectors."""
    intersection = set(vec1.keys()).intersection(set(vec2.keys()))
    numerator = sum(vec1[x] * vec2[x] for x in intersection)

    sum1 = sum(v ** 2 for v in vec1.values())
    sum2 = sum(v ** 2 for v in vec2.values())
    denominator = math.sqrt(sum1) * math.sqrt(sum2)

    if denominator == 0:
      return 0.0
    return numerator / denominator

  def fit(self, documents):
    """Performs single-pass clustering on a dict of {doc_id: text}."""
    for doc_id, text in documents.items():
      doc_vec = self._text_to_vector(text)

      if not self.clusters:
        # First document initializes the first cluster
        self.clusters.append(doc_vec)
        self.cluster_docs.append([doc_id])
        continue

      # Find similarity with all existing cluster centroids
      similarities = [
          self._cosine_similarity(doc_vec, centroid)
          for centroid in self.clusters
      ]
      max_sim = max(similarities)
      best_cluster_idx = similarities.index(max_sim)

      if max_sim >= self.threshold:
        # Assign to existing cluster and update centroid (simple merge/average)
        self.cluster_docs[best_cluster_idx].append(doc_id)
        # Update centroid vector
        for term, freq in doc_vec.items():
          self.clusters[best_cluster_idx][term] += freq
      else:
        # Create a new cluster
        self.clusters.append(doc_vec)
        self.cluster_docs.append([doc_id])

    return self.cluster_docs


In [6]:
# --- Example Usage (Consider 5 files/documents) ---
documents = {
    "Doc1": "machine learning and artificial intelligence systems",
    "Doc2": "deep learning neural networks and machine learning",
    "Doc3": "information retrieval and web search engines",
    "Doc4": "search algorithms and text information indexing",
    "Doc5": "data mining algorithms and machine learning models",
}

# Run Single-Pass Clustering
spc = SinglePassClustering(threshold=0.2)
clusters = spc.fit(documents)

# Display Results
print("--- Single-Pass Clustering Results ---")
for i, cluster in enumerate(clusters):
  print(f"Cluster {i + 1}: {cluster}")

--- Single-Pass Clustering Results ---
Cluster 1: ['Doc1', 'Doc2', 'Doc5']
Cluster 2: ['Doc3', 'Doc4']
